# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sujan-lab-cell/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

> **Which webpages should an SEO/content team review first when many pages may be experiencing search-performance decline?**

The practical challenge is that large content portfolios can contain thousands of pages, making it difficult to manually decide which pages deserve attention first.

This project asks whether historical search-performance signals can be used to **rank pages for human review** based on their likelihood of meeting the defined May 2026 click-decline outcome.

The goal is not to automatically decide which pages should be refreshed. Instead, the model provides **decision-support prioritization** so that SEO and content teams can focus their limited review time on higher-ranked pages.

In [ ]:
import json
from pathlib import Path
from urllib.request import urlopen

# Load Week 7 metrics artifact
path = Path("work/outputs/ml_action_playbook_metrics.json")

if path.exists():
    with open(path, "r", encoding="utf-8") as f:
        metrics = json.load(f)
else:
    url = "https://raw.githubusercontent.com/Sujan-lab-cell/flyrank-ml-internship/main/work/outputs/ml_action_playbook_metrics.json"
    with urlopen(url) as response:
        metrics = json.load(response)

# Extract metrics
evaluation_population = metrics["evaluation_population"]
unique_clients = metrics["unique_clients"]
validation_method = metrics["validation"]["method"]
precision_at_50_random_forest = metrics["validation"]["precision_at_50_random_forest"]
precision_at_50_baseline = metrics["validation"]["precision_at_50_baseline"]

# Assertions
assert evaluation_population == 16513
assert unique_clients == 36
assert precision_at_50_random_forest == 0.444
assert precision_at_50_baseline == 0.392

# Print metrics
print("Evaluation population:", evaluation_population)
print("Unique clients:", unique_clients)
print("Validation method:", validation_method)
print("Random Forest Precision@50:", precision_at_50_random_forest)
print("Baseline Precision@50:", precision_at_50_baseline)
print("Decision supported: rank pages for human review based on potential click decline.")

Evaluation population: 16513
Unique clients: 36
Validation method: 5-fold GroupKFold by client
Random Forest Precision@50: 0.444
Baseline Precision@50: 0.392
Decision supported: rank pages for human review based on potential click decline.


## 2. Data

> **The model uses historical FlyRank search-performance data available before the May 2026 outcome period.**

### Data Source

The analysis uses the **FlyRank full warehouse release** and focuses on search-performance history at the content-page level.

The main data sources were:

- `fact_daily` — daily search-performance records.
- `dim_content` — content-page information.
- `dim_clients` — client/group information.

### Date Windows

The model uses:

- **February 1 – April 30, 2026** → feature and training history.
- **May 1 – May 31, 2026** → outcome window used to define click decline.

May data was used only for the outcome label and was not used as a model feature.

### Eligible Population

A page was included only when:

- `impressions_total >= 1000`
- `april_clicks >= 10`

This produced **16,513 eligible pages across 36 clients**.

### What Was Excluded

Some fields were excluded from the model because they were not suitable for the final task:

- **Future May metrics** — would leak information from the outcome period.
- **Target-related fields** such as `may_clicks` — used only to create the label.
- **Client and content IDs** — used for grouping, joining, and deterministic tie-breaking, not as predictive features.
- **Sparse session, scroll, and AI-referral signals** — excluded because their coverage was limited for this modeling population.

### Public-Safety

The paper reports aggregate results only. It does not expose client names, private URLs, private search queries, credentials, or raw production exports.

In [ ]:
import json
from pathlib import Path
from urllib.request import urlopen

# Load Week 7 metrics artifact (local path with raw GitHub URL fallback)
path = Path("work/outputs/ml_action_playbook_metrics.json")
if path.exists():
    with open(path, "r", encoding="utf-8") as f:
        metrics = json.load(f)
else:
    url = "https://raw.githubusercontent.com/Sujan-lab-cell/flyrank-ml-internship/main/work/outputs/ml_action_playbook_metrics.json"
    with urlopen(url) as response:
        metrics = json.load(response)

# Extract Data section facts from artifact
evaluation_population = metrics["evaluation_population"]
unique_clients = metrics["unique_clients"]
eligibility_criteria = metrics["eligibility_criteria"]
target_definition = metrics["target_definition"]

# Assertions verifying documented population and criteria
assert evaluation_population == 16513
assert unique_clients == 36
assert eligibility_criteria == "impressions_total >= 1000 AND april_clicks >= 10"
assert target_definition == "decline = (may_clicks < 0.8 * april_clicks).astype(int)"

# Print verified Data section facts
print("Evaluation population:", evaluation_population)
print("Unique clients:", unique_clients)
print("Eligibility criteria:", eligibility_criteria)
print("Target definition:", target_definition)


Evaluation population: 16513
Unique clients: 36
Eligibility criteria: impressions_total >= 1000 AND april_clicks >= 10
Target definition: decline = (may_clicks < 0.8 * april_clicks).astype(int)


## 3. Methodology

### Target Definition

A page was labeled as **declining** when its May 2026 clicks were less than 80% of its April 2026 clicks:

`decline = (may_clicks < 0.8 × april_clicks)`

This defines the prediction target used throughout the experiment.

### Features

The model used historical information available before the May outcome period:

- impressions_total
- clicks_total
- april_impressions
- april_clicks
- feb_clicks
- momentum
- CTR
- active_days
- weighted_position

May performance metrics were not used as predictive features.

### Baseline

The baseline flagged a page when:

`april_clicks < march_clicks`

Flagged pages were ranked using April impressions.

This provides a simple reference point for evaluating whether the machine-learning model adds useful ranking signal.

### Model

The main model was a **Random Forest classifier**.

The model was evaluated using the same eligible population, target definition, ranking procedure, and evaluation metrics as the baseline.

### Validation Design

Performance was evaluated using **5-fold GroupKFold by client**.

Client identity was used only to define the groups, so pages from the same client did not appear in both the training and validation portions of a fold.

This tests whether the model can generalize beyond the clients used to train it.

### Leakage Checks

A validation audit checked that:

- future May outcome fields were not used as features
- target-related fields were excluded from the feature matrix
- client and content IDs were not used as predictive features
- all model features ended at or before the April 30, 2026 cutoff

The Week 6 validation audit reported that no future or target-related fields were used as model features. The capstone notebook does not independently rerun the full leakage audit because it relies on the recorded project artifacts.

In [4]:
import json
from pathlib import Path
from urllib.request import urlopen

# Load Week 7 metrics artifact (local path with raw GitHub URL fallback)
path = Path("work/outputs/ml_action_playbook_metrics.json")
if not path.exists():
    path = Path("../outputs/ml_action_playbook_metrics.json")
if path.exists():
    with open(path, "r", encoding="utf-8") as f:
        metrics = json.load(f)
else:
    url = "https://raw.githubusercontent.com/Sujan-lab-cell/flyrank-ml-internship/main/work/outputs/ml_action_playbook_metrics.json"
    with urlopen(url) as response:
        metrics = json.load(response)

# Extract and define methodology facts
evaluation_population = metrics["evaluation_population"]
unique_clients = metrics["unique_clients"]
target_definition = metrics["target_definition"]
validation_method = metrics["validation"]["method"]
client_overlap = metrics["validation"]["client_overlap_per_fold"]
baseline_definition = "april_clicks < march_clicks, ranked by April impressions"
main_model = "Random Forest"
model_features = [
    "impressions_total", "clicks_total", "april_impressions", "april_clicks",
    "feb_clicks", "momentum", "CTR", "active_days", "weighted_position"
]

# Check if leakage audit result is recorded in the JSON artifact
leakage_audit_result = metrics.get(
    "leakage_audit",
    metrics.get(
        "leakage_audit_passed",
        metrics.get("validation", {}).get("leakage_audit"),
    ),
)
if leakage_audit_result is None:
    leakage_audit_status = "Cannot be independently verified from available artifact"
else:
    leakage_audit_status = (
        "Passed" if leakage_audit_result is True else str(leakage_audit_result)
    )

# Assertions verifying documented methodology facts available in artifacts
assert evaluation_population == 16513
assert unique_clients == 36
assert target_definition == "decline = (may_clicks < 0.8 * april_clicks).astype(int)"
assert validation_method == "5-fold GroupKFold by client"
assert client_overlap == [0, 0, 0, 0, 0]
assert sum(client_overlap) == 0
assert len(model_features) == 9
assert "precision_at_50_random_forest" in metrics["validation"]
if leakage_audit_result is not None:
    assert leakage_audit_result is True or leakage_audit_result == "Passed"

# Print verified methodology facts
print("Target definition:", target_definition)
print("Model features (9):", ", ".join(model_features))
print("Main model:", main_model)
print("Baseline definition:", baseline_definition)
print("Validation design:", validation_method)
print("Client overlap per fold:", client_overlap, "(0 overlap)")
print("Leakage audit:", leakage_audit_status)
print("Population:", evaluation_population, "eligible pages across", unique_clients, "clients")


Target definition: decline = (may_clicks < 0.8 * april_clicks).astype(int)
Model features (9): impressions_total, clicks_total, april_impressions, april_clicks, feb_clicks, momentum, CTR, active_days, weighted_position
Main model: Random Forest
Baseline definition: april_clicks < march_clicks, ranked by April impressions
Validation design: 5-fold GroupKFold by client
Client overlap per fold: [0, 0, 0, 0, 0] (0 overlap)
Leakage audit: Cannot be independently verified from available artifact
Population: 16513 eligible pages across 36 clients


## 4. Results (vs baseline)

The Random Forest was compared with the same simple baseline using the same eligible population, client-grouped validation design, ranking procedure, and Precision@K evaluation.

| Method | Precision@10 | Precision@20 | Precision@50 | Precision@100 |
|---|---:|---:|---:|---:|
| Baseline | 0.400 | 0.370 | 0.392 | 0.388 |
| Random Forest | 0.460 | 0.430 | 0.444 | 0.448 |

The Random Forest achieved a **Precision@50 of 0.444**, compared with **0.392 for the baseline**, an improvement of **5.2 percentage points**.

This provides directional evidence that the model can improve the ranking of potentially declining pages for human review under the evaluated client-grouped validation setup.

The result should not be treated as a production guarantee or service-level target. Performance varied across validation folds, so additional validation would be needed before applying the model to new clients or changing the decision process.

In [ ]:
import json
from pathlib import Path
from urllib.request import urlopen
import matplotlib.pyplot as plt

# Load Week 7 metrics artifact (local path with raw GitHub URL fallback)
path = Path("work/outputs/ml_action_playbook_metrics.json")
if not path.exists():
    path = Path("../outputs/ml_action_playbook_metrics.json")
if path.exists():
    with open(path, "r", encoding="utf-8") as f:
        metrics = json.load(f)
else:
    url = "https://raw.githubusercontent.com/Sujan-lab-cell/flyrank-ml-internship/main/work/outputs/ml_action_playbook_metrics.json"
    with urlopen(url) as response:
        metrics = json.load(response)

# Extract Precision@K values
pk_base = metrics["precision_at_k"]["baseline"]
pk_rf = metrics["precision_at_k"]["random_forest"]

base_p10 = pk_base["P@10"]
base_p20 = pk_base["P@20"]
base_p50 = pk_base["P@50"]
base_p100 = pk_base["P@100"]

rf_p10 = pk_rf["P@10"]
rf_p20 = pk_rf["P@20"]
rf_p50 = pk_rf["P@50"]
rf_p100 = pk_rf["P@100"]

# Assertions for all eight values
assert base_p10 == 0.400
assert base_p20 == 0.370
assert base_p50 == 0.392
assert base_p100 == 0.388

assert rf_p10 == 0.460
assert rf_p20 == 0.430
assert rf_p50 == 0.444
assert rf_p100 == 0.448

# Calculate P@50 improvement in percentage points
p50_improvement_pp = round((rf_p50 - base_p50) * 100, 1)
assert p50_improvement_pp == 5.2

# Print verified metrics
print(f"Baseline Precision@K: P@10={base_p10:.3f}, P@20={base_p20:.3f}, P@50={base_p50:.3f}, P@100={base_p100:.3f}")
print(f"Random Forest Precision@K: P@10={rf_p10:.3f}, P@20={rf_p20:.3f}, P@50={rf_p50:.3f}, P@100={rf_p100:.3f}")
print(f"P@50 Improvement: {p50_improvement_pp} percentage points")

# Plot Precision@K Comparison Chart
k_values = [10, 20, 50, 100]
base_scores = [base_p10, base_p20, base_p50, base_p100]
rf_scores = [rf_p10, rf_p20, rf_p50, rf_p100]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(k_values, rf_scores, marker="o", label="Random Forest")
ax.plot(k_values, base_scores, marker="s", linestyle="--", label="Baseline")

ax.set_xlabel("K")
ax.set_ylabel("Precision@K")
ax.set_title("Precision@K Comparison: Random Forest vs Baseline")
ax.set_xticks(k_values)
ax.set_ylim(0, 1)
ax.grid(True)
ax.legend()

# Save chart
fig_dir = Path("work/figures")
if not fig_dir.parent.exists() and Path("../figures").parent.exists():
    fig_dir = Path("../figures")
fig_dir.mkdir(parents=True, exist_ok=True)
fig_path = fig_dir / "ml_precision_at_k_comparison.png"

plt.savefig(fig_path, bbox_inches="tight", dpi=300)

plt.close(fig)

## 5. Limitations

*What this work cannot claim.*

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
